# Использование сокета

Сокет — это программный интерфейс (endpoint) для обмена данными между процессами по сети или внутри одной машины.

Сокеты упрощают разработчикам работу по передаче сообщений: они инкапсулируют внутри себя всю логику работы с сетью и передачей пакетов данных.

Внутри системы каждый IP-адрес принадлежит какому-нибудь сетевому интерфейсу. К сетевым интерфейсам могут относить:
- Физические интерфейсы. Порты в компьютерах, Wi-Fi-адаптеры и прочие устройства. Обычно на Linux-системах они называются eno1, enp0s3 для ethernet-соединения и wlan0 или wlp5s0 для Wi-Fi.
- Виртуальные интерфейсы. Существуют только на уровне операционной системы и не привязаны к физическому адаптеру. Первичный виртуальный интерфейс — петлевой (loopback interface). Он используется для передачи данных по сети между процессами внутри одного компьютера. Стандартно использует подсеть 127.0.0.0/8 (адреса от 127.0.0.0 до 127.255.255.255). Такие соединения доступны только внутри одного физического компьютера и не видны внешним клиентам. В этот диапазон входит и известный вам localhost (127.0.0.1) — адрес обратной связи с компьютером.

In [7]:
# Простой HTTP-сервер на сокетах, который обрабатывает запросы в одном потоке
# Откройте в браузере http://localhost:8888/ и увидите, что сервер возвращает вам в ответе текст запроса, который вы отправили.
# В терминале отправьте запрос curl -X POST http://localhost:8888/
import socket

HOST, PORT = '', 8888


def handle_request(request: bytes) -> bytes:
    """
    Простая функция для обработки HTTP-запроса и формирования ответа.
    Она декодирует байты запроса в строку, формирует HTTP-ответ,
    который включает в себя текст запроса, и возвращает его в виде байтов
    """
    request_data = request.decode()
    http_response = f"""HTTP/1.1 200 OK\nContent-Type: text/html\n\n{request_data}"""
    return http_response.encode()


def serve_forever():
    # Создаём сокет для обмена данными между процессами
    # где socket.AF_INET - использовать IPv4-адресацию, socket.SOCK_STREAM - использовать протокол TCP
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as listen_socket:
        # Устанавливаем опцию SO_REUSEADDR, чтобы сокет мог быть повторно использован после закрытия
        listen_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, True)
        # Привязываем сокет к заданному адресу и порту
        listen_socket.bind((HOST, PORT))
        # Переводим сокет в режим прослушивания, чтобы он мог принимать запросы от других процессов
        listen_socket.listen()
        print('Serving HTTP on port {port} ...'.format(port=PORT))

        # Бесконечный цикл для обработки входящих запросов
        while True:
            # Ждём, пока кто-то не присоединится к серверу и получим соединение и адрес клиента
            client_connection, client_address = listen_socket.accept()
            # Используем объект клиентского сокета для получения данных от клиента и отправки ответа
            with client_connection:
                # Получаем данные от клиента пачками по 1024 байта
                request = client_connection.recv(1024)
                # Обрабатываем запрос
                http_response = handle_request(request)
                # Отправляем ответ клиенту
                client_connection.sendall(http_response)


if __name__ == '__main__':
    try:
        serve_forever()
    except KeyboardInterrupt:
        print("\nСервер остановлен пользователем")

Serving HTTP on port 8888 ...

Сервер остановлен пользователем


Сокет использует для своей работы файловые дескрипторы (это просто число, которое ОС выдаёт процессу для работы с ресурсом ввода-вывода), но в некоторых случаях ему может не хватить объёма портов — максимум 65535. Поэтому необходимо включить опцию переиспользования портов, которая позволит подключать больше клиентов, чем портов, так как файловые дескрипторы не имеют ограничения по количеству.

Под капотом сокеты работают с событиями операционной системы. Можно создать очень много сокетов и опрашивать их по кругу каждый раз, но при множестве соединений такой вариант не будет эффективным. Но есть способ сократить нагрузку на ОС — поллинг. Это метод отслеживания новых событий ОС и реакции на них или, другими словами, способ обработки событий в событийно-ориентированных системах. 

Часть программы, которая умеет правильно обрабатывать события по сокетам от ОС, называется селектором. Это механизм, который позволяет ждать событий от нескольких файловых дескрипторов одновременно в одном потоке

У селектора существует два основных события для обработки:
- EVENT_READ (1) — событие, когда сокет готов принимать данные.
- EVENT_WRITE (2) — событие, когда данные были отправлены из сокета или когда сокет снова готов для записи данных.

In [9]:
# Простой TCP-сервер на сокетах, который обрабатывает несколько клиентов в одном потоке с помощью селектора
# Селектор позволяет эффективно управлять несколькими соединениями, не блокируя выполнение программы при ожидании данных от клиентов.
# Для проверки выполните curl -X POST http://localhost:8000/ и увидите, что сервер возвращает вам в ответе текст запроса, который вы отправили.
import logging
import selectors
import socket
import sys

HOST, PORT = '', 8000  # Порт сервера

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(stream=sys.stdout))


def new_connection(selector: selectors.BaseSelector, sock: socket.socket):
    """
    Метод для обработки новых входящих соединений.
    Он вызывается, когда серверный сокет получает событие чтения, что означает, что кто-то пытается подключиться к серверу
    """
    # Ждём, пока кто-то не присоединится к серверу и получим соединение и адрес клиента
    # Создаётся новый сокет для общения с клиентом и получаем его адрес
    new_conn, address = sock.accept()
    logger.info('accepted new_conn from %s', address)
    # Используем объект клиентского сокета для получения данных от клиента и отправки ответа
    new_conn.setblocking(False)

    # Регистрируем новый клиентский сокет в селекторе для отслеживания событий чтения (когда клиент отправляет данные)
    selector.register(new_conn, selectors.EVENT_READ, read_callback)


def read_callback(selector: selectors.BaseSelector, sock: socket.socket):
    """
    Метод для обработки событий чтения на клиентском сокете. Он вызывается, когда клиент отправляет данные на сервер.
    """
    # Получаем данные от клиента пачками по 1024 байта
    data = sock.recv(1024)
    if data:
        # Обрабатываем запрос и формируем ответ (в данном случае просто отправляем обратно полученные данные)
        sock.send(data)
    else:
        logger.info('closing connection %s', sock)
        # Если же данных нет (data пустой), это означает, что клиент закрыл соединение, и сервер выполняет очистку:
        # выводит сообщение о закрытии соединения, удаляет сокет из селектора с помощью метода unregister() и закрывает сокет с помощью метода close().
        selector.unregister(sock)
        sock.close()


def run_iteration(selector: selectors.BaseSelector):
    """
    Метод выполняет одну итерацию обработки событий в селекторе.
    Он вызывается в бесконечном цикле внутри метода serve_forever для постоянного прослушивания новых сообщений от клиентов.
    """
    # Получаем список событий, которые произошли на зарегистрированных сокетах
    events = selector.select()
    for key, mask in events:
        # Получаем callback-функцию, которая была зарегистрирована вместе с сокетом
        # (new_connection для серверного сокета и read_callback для клиентского сокета)
        callback = key.data
        # Получаем сокет, на котором произошло событие
        socket = key.fileobj
        # Вызываем callback-функцию, передавая ей селектор (selector) и сокет (key.fileobj), на котором произошло событие
        callback(selector, socket)


def serve_forever():
    """
    Метод запускает сервер на постоянное прослушивание новых сообщений
    """
    # Создаём селектора (системный механизм ожидания готовности файловых дескрипторов)
    with selectors.SelectSelector() as selector:
        # Создаём сокет (интерфейс для обмена данными между процессами)
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:
            # Устанавливаем опцию SO_REUSEADDR, чтобы сокет мог быть повторно использован после закрытия
            server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, True)
            # Привязываем сокет к заданному адресу и порту
            server_socket.bind((HOST, PORT))
            # Переводим сокет в режим прослушивания, чтобы он мог принимать запросы от других процессов
            server_socket.listen()
            # Устанавливаем неблокирующий режим для сокета, чтобы он не блокировал выполнение программы при ожидании входящих соединений
            server_socket.setblocking(False)
            logger.info('Server started on port %s', PORT)

            # Регистрируем серверный сокет в селекторе для отслеживания событий чтения (новые соединения)
            # и связываем его с callback-функцией new_connection для обработки новых входящих соединений
            selector.register(server_socket, selectors.EVENT_READ, new_connection)

            while True:
                # На каждой итерации бесконечного цикла вызываем метод run_iteration для обработки событий, которые произошли на зарегистрированных сокетах
                run_iteration(selector)


if __name__ == '__main__':
    try:
        serve_forever()
    except KeyboardInterrupt:
        print("\nСервер остановлен пользователем")

Server started on port 8000
Server started on port 8000
Server started on port 8000
Server started on port 8000
Server started on port 8000
Server started on port 8000

Сервер остановлен пользователем
